<a href="https://colab.research.google.com/github/thiru-bit/SP25-690-Gugulothu/blob/main/Sentiment_Analysis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 'transformers' is for BERT, 'datasets' is to get the movie reviews
!pip install transformers datasets scikit-learn

In [ ]:
import torch # The brain of the project
import numpy as np
from datasets import load_dataset  # The library to get the movie reviews
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments  # The BERT tools
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
# Download the movie reviews(loading the data)
dataset = load_dataset("imdb")
dataset

In [ ]:
# We take 2,000 reviews for training and 500 for testing/validation
# This makes sure our 'LEGO castle' is small enough to build quickly!
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(f"New training size: {len(small_train_dataset)}")
print(f"New testing size: {len(small_test_dataset)}")

In [ ]:
##TOKENIZATION (convert text → numbers)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

train_data = small_train_dataset.map(tokenize_function, batched=True)
test_data = small_test_dataset.map(tokenize_function, batched=True)

In [ ]:
#FORMAT DATA FOR MODEL
train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
#LOAD BERT MODEL
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
#DEFINE METRICS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
#TRAINING SETTINGS
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
)

In [ ]:
#TRAIN MODEL
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
#EVALUATE MODEL
trainer.evaluate()

In [ ]:
#SAVE MODEL
trainer.save_model("bert_sentiment_model")

In [ ]:
#TESTING OUR MODEL
text = "This movie was absolutely amazing!"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to("cuda") for k, v in inputs.items()}
outputs = model(**inputs)
prediction = torch.argmax(outputs.logits, dim=1)

print("Prediction:", "Positive" if prediction.item() == 1 else "Negative")

In [ ]:
# LSTM BASELINE MODEL IMPLEMENTATION

# 1. IMPORTS
# ==============================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score
import re

# ==============================
# 2. LOAD DATASET
# ==============================
dataset = load_dataset("imdb")

# Use smaller subset for speed
train_data = dataset["train"].shuffle(seed=42).select(range(5000))
test_data = dataset["test"].shuffle(seed=42).select(range(5000))

# ==============================
# 3. TEXT CLEANING + VOCAB
# ==============================
def clean(text):
    return re.sub(r"[^a-zA-Z ]", "", text.lower()).split()

vocab = {}
index = 2  # 0=pad, 1=unk

for sample in train_data:
    for word in clean(sample["text"]):
        if word not in vocab:
            vocab[word] = index
            index += 1

def encode(text, max_len=200):
    tokens = clean(text)
    ids = [vocab.get(t, 1) for t in tokens][:max_len]
    ids += [0] * (max_len - len(ids))
    return ids

# ==============================
# 4. DATASET CLASS
# ==============================
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = encode(self.data[idx]["text"])
        label = self.data[idx]["label"]
        return torch.tensor(text), torch.tensor(label)

# ==============================
# 5. DATALOADERS
# ==============================
train_loader = DataLoader(IMDBDataset(train_data), batch_size=32, shuffle=True)
test_loader = DataLoader(IMDBDataset(test_data), batch_size=32)

# ==============================
# 6. MODEL (FIXED)
# ==============================
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab)+2, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        h = self.dropout(h[-1])
        return self.fc(h)

# ==============================
# 7. SETUP
# ==============================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LSTMModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ==============================
# 8. TRAINING
# ==============================
for epoch in range(3):
    model.train()

    for x, y in train_loader:
        x, y = x.to(device), y.float().to(device)

        preds = model(x).squeeze()

        # DEBUG: check predictions
        if epoch == 0:
            print("Sample raw preds:", preds[:5].detach().cpu())

        loss = criterion(preds, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} done")

# ==============================
# 9. EVALUATION
# ==============================
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)

        preds = model(x).squeeze()
        preds = torch.sigmoid(preds)
        preds = (preds > 0.5).int()

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.tolist())

# ==============================
# 10. DEBUG CHECKS
# ==============================
print("Unique predictions:", set(all_preds))
print("First 10 preds:", all_preds[:10])
print("First 10 labels:", all_labels[:10])

# Label balance check
print("Label distribution:", sum(all_labels)/len(all_labels))

# ==============================
# 11. FINAL ACCURACY
# ==============================
acc = accuracy_score(all_labels, all_preds)
print("✅ LSTM Test Accuracy:", acc)